This notebook might not be usable "out of the box" and is mainly meant to give an idea how the data was calculated from the NMR bundles and the MD ensembles.

# Import modules

In [1]:
import numpy as np
import pandas as pd
import pickle as pkl
import barnaba as bb
from barnaba import definitions, functions
from subprocess import Popen, PIPE
import regex as re
import mdtraj as md
from lib.noe_helper import *

In [2]:
%load_ext autoreload
%autoreload 2
import os
import sys
module_path = os.path.abspath(os.path.join('lib')) # or the path to your source code
sys.path.insert(0, module_path)
from lib.noe_helper import *

# Define Functions

In [3]:
def load( inp ):
    pin = open( inp, "rb" )
    return pkl.load( pin )

def save( outfile, results ):
    with open( outfile + ".pkl", "wb" ) as fp:
        pkl.dump( results, fp )

In [4]:
### HELPER FUNCTIONS TO CALCULATE NOE DISTANCES ###

# helper functions, subsititute strings. This is because the name of hydrogens is a mess
# alt = {"H2'":"1H2'","H5''":"2H5'","H5'":"1H5'","HO2'":"2HO'","H5\"":"2H5'","H5'2":"2H5'","H5'1":"1H5'"}
alt = {"H2'":"H2'1","H5''":"H5'2","H5'":"H5'1","HO2'":"HO'2"}

def sub(ss):
    at = "H" + ss.split("H")[1]
    if(at in alt):
        at = alt[at]
    return ss.split("H")[0] + at

# read experimental datafile and returns a list of labels and experimental values
def read_exp(f_exp):
    labels = []
    vals = []
    with open(f_exp) as fh:
        for line in fh:
            if("#" not in line):
                r1 = line.split()[0].split("-")[0]
                print(r1)
                r2 = line.split()[0].split("-")[1]
                print(r2)
                v1 = np.sort([r1,r2])
                print(v1)
                qq = v1[0] +"/"+ v1[1]
                
                if(qq in labels):
                    print("# DUPLICATE. Skipping data.."),
                    print(qq,vals[labels.index(qq)], line),
                else:
                    vals.append([float(line.split()[1]),float(line.split()[2])])
                    labels.append(qq)
    return labels,vals

# get labels from df column (Assignment) 
def get_labels(df, col):
    labels = []
    for asm in df.iloc[:,col]:
        r1 = asm.split()[0].split("-")[0]
        r2 = asm.split()[0].split("-")[1]
        v1 = np.sort([r1,r2])
        qq = v1[0] +"/"+ v1[1]
        labels.append(qq)
    return labels

# find indeces in topology corresponding to labels in experimental datafile
def get_idxs(labels,top):

    atoms = []
    for atom in top.atoms:
        aa = str(atom).split("-")[1]
        if(aa in alt): aa = alt[aa]
        atoms.append("%s%s" % (str(atom).split("-")[0],aa))
    pairs = []
    for el in labels:
        ss  = el.split("/")
        at1 = sub(ss[0])
        at2 = sub(ss[1])
        if(at1 in atoms and at2 in atoms):
            pairs.append([atoms.index(at1),atoms.index(at2)])
        else:
            print("# Warning: Either %s or %s are missing" % (at1,at2))
            return 0
    if len(pairs) != len(labels):
        print("# Found only %d pairs out of %d" % (len(pairs),len(labels)))
    return np.array(pairs)

def group_by_heading( some_source ):
    buffer= []
    for line in some_source:
        if line.startswith( " ASSI" ):
            if buffer: yield buffer
            buffer= [ line.strip().strip(')') ]
        else:
            buffer.append( line.strip().strip(')') )
    yield buffer

# From NMR bundles

Back-calculate experimental observables from

Since some of the forward models need certain PDB formats we have to reformat the PDB files.

In [5]:
# NMR bundle directory:
bundles_dir = 'nmrbundle_data'

# Experimental Results directory
exp_data_dir = "exp_data"

# Bundle identifiers:
bundles = ['A', 'Farfar1', 'Farfar2', 'alphafold3', "2F87"] #literature pdb 2F87 needs to be last, due to certain special treamtment due to different stem architecture


## RDCs

We use the pf1-phage prediction method implemented in *PALES** to back-calculate RDCs. In order to do that we use the script ```calc_rdc_bundles.py``` which submits each frame to *PALES*, predicts the alignment tensor and calculates RDCs, parses the output and saves the D values in a Pickle file.

The actual command line to run *PALES is*: ```pales-linux -inD exp_data/exp_rdc_pales.tab -pdb {pdb_tmp} -outD {outd_tmp} -pf1 -H -wv 0.05```


*Zweckstetter, M. NMR: Prediction of molecular alignment from structure using the PALES software. Nat. Protoc. 3, 679–690 (2008).

In [6]:
 # RDCs (works)
rdc_exp_input = pd.read_csv(f'{exp_data_dir}/exp_rdc_pales.tab', names=['RESID_I', 'RESNAME_I', 'ATOMNAME_I', 'RESID_J', 'RESNAME_J',
       'ATOMNAME_J', 'D', 'DD', 'W'], delim_whitespace=True, skiprows=5)
# all residues:
rdc_exp_labels_all = [f"{row[1]['RESNAME_I'][0]}{row[1]['RESID_I']}_{row[1]['ATOMNAME_I']}-{row[1]['ATOMNAME_J']}" for row in rdc_exp_input.iterrows()]
# loop residues:
rdc_exp_labels_loop = [f"{row[1]['RESNAME_I'][0]}{row[1]['RESID_I']}_{row[1]['ATOMNAME_I']}-{row[1]['ATOMNAME_J']}" for row in rdc_exp_input.iterrows() if row[1]['RESID_I'] in [5,6,7,8,9,10]]

In [11]:
for bundle in bundles:
    process = Popen(f'python {bundles_dir}/calc_rdc_bundles.py {bundle}', shell=True, stdin=PIPE, stdout=PIPE, universal_newlines=True)
    process.wait() # needed to run one process at a time (because of using temporary files etc.)

Parse the output and write BME readable files:

In [12]:
for bundle in bundles:
    d_df = load(f'{bundles_dir}/nmrbundle_{bundle}_d_df_pf1.pkl')
    for i in rdc_exp_labels_all: # missing residues in 1RNGaxis=1
        if i not in d_df.columns:
            d_df[i] = np.full((len(d_df)), np.nan)
    
    # All res:
    d_df[rdc_exp_labels_all].to_csv(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme.dat', header=False, sep='\t', na_rep=np.nan)
    # Loop only:
    d_df[rdc_exp_labels_loop].to_csv(f'{bundles_dir}/nmrbundle_{bundle}_rdc_loop_bme.dat', header=False, sep='\t', na_rep=np.nan)

The predicted RDCs need to be scaled to the experimental values before comparing them. To reduce the effect of over-fitting we scale the RDCs based on all measurements and since we treat NMR structures as a bundle of structures rather than an ensemble we do the scaling for all individual structures.

In [ ]:
for bundle in bundles:
    rdc_bundle = np.loadtxt(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme.dat')
    L = np.zeros(len(rdc_bundle))
    for i in range(len(L)):
        masked = np.ma.array(rdc_bundle[i,1:], mask=np.isnan(rdc_bundle[i,1:])) # mask NaNs (only present in 2F87 due to missing/different residues in the stem)
        L[i] = np.sum(np.array(rdc_exp_input['D'])*masked)/np.sum(masked*masked)
    np.save(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme_L', L)

## $^3$J-couplings

We use Barnaba* to calculate the $^3$J scalar couplings 2H5H4, H1H2, H2H3, H3P, C4Pb, 1H5P, 1H5H4, C4Pe, 2H5P and H3H4.

Definition: $A cos^2 (\theta + \phi) + B cos (\theta + \phi) + C$

*Bottaro, S. et al. Barnaba: software for analysis of nucleic acid structures and trajectories. Rna 25, 219–231 (2019)

In [14]:
# A (Hz), B (Hz), C (Hz), _, phi (rad)
bb.definitions.couplings_karplus

{'H1H2': [9.67, -2.03, 0.0, 0.0, 0.0],
 'H2H3': [9.67, -2.03, 0.0, 0.0, 0.0],
 'H3H4': [9.67, -2.03, 0.0, 0.0, 0.0],
 '1H5P': [15.3, -6.1, 1.6, 0.0, -2.094395],
 '2H5P': [15.3, -6.1, 1.6, 0.0, 2.094395],
 'C4Pb': [6.9, -3.4, 0.7, 0.0, 0.0],
 '1H5H4': [9.7, -1.8, 0.0, 0.0, -2.094395],
 '2H5H4': [9.7, -1.8, 0.0, 0.0, 0.0],
 'H3P': [15.3, -6.1, 1.6, 0.0, 2.094395],
 'C4Pe': [6.9, -3.4, 0.7, 0.0, 0.0],
 'H1C2/4': [4.7, 2.3, 0.1, 0.0, -1.0471975],
 'H1C6/8': [4.5, -0.6, 0.1, 0.0, -1.0471975]}

In [ ]:
j3_exp_labels = list(pd.read_csv('exp_data/exp_j3_bme.dat', delim_whitespace=True).index) # ONLY EXTENDED-LOOP RESIDUES!!!
j3_exp_couplings = list(set([ i.split('-')[-1] for i in j3_exp_labels ]))

# Never used?
barnaba_to_exp = {'H1H2':"H1',H2'", 'H2H3':"H2',H3'", 'H3H4':"H3',H4'",\
                '1H5P':"H5'i,Pi", '2H5P':"H5''i,Pi",\
                'C4Pb':"C4'i,Pi", '1H5H4':"NA", '2H5H4':"NA", \
                'H3P':"H3'i,Pi+1" ,'C4Pe':"C4'i,Pi+1", \
                'H1C2/4':"NA", 'H1C6/8':"NA"}

# the shape of couplings is (nframes, nresidues, ncouplings)
# only look at bundle A
for bundle in bundles[:-1]:
    traj_file   = f'{bundles_dir}/Bundle_{bundle}.pdb'
    top_file    = f'{bundles_dir}/Bundle_{bundle}.pdb'

    couplings,residues = bb.jcouplings(traj_file, topology=top_file, couplings=j3_exp_couplings)
    print( f'Calculated jcouplings for {j3_exp_couplings}.' )
    print( couplings.shape )

    couplings_avg = np.average( couplings, 0 )
    print(couplings_avg.shape)
    
    j3_calc_ = pd.DataFrame()
    for i,ii in enumerate([ i.split('_')[0]+i.split('_')[1] for i in residues]):
        for j,jj in enumerate(j3_exp_couplings):
            print(f'Bundle {traj_file} Coupling: {ii}-{jj}')
            j3_calc_[f'{ii}-{jj}'] = couplings[:,i,j]

    ll = []
    for i in j3_exp_labels:
        ll.append(j3_calc_[i])
    pd.DataFrame(ll).T.to_csv(f'{bundles_dir}/nmrbundle_{bundle}_j3_loop_bme.dat', sep=' ', header=False)

# Loading nmrbundle_data/Bundle_A.pdb 


Calculated jcouplings for ['H3H4', 'H3P', '2H5P', 'H1H2', 'C4Pe', '1H5P', 'H2H3'].
(20, 14, 7)
(14, 7)
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G1-H3H4
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G1-H3P
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G1-2H5P
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G1-H1H2
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G1-C4Pe
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G1-1H5P
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G1-H2H3
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G2-H3H4
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G2-H3P
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G2-2H5P
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G2-H1H2
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G2-C4Pe
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G2-1H5P
Bundle nmrbundle_data/Bundle_A.pdb Coupling: G2-H2H3
Bundle nmrbundle_data/Bundle_A.pdb Coupling: C3-H3H4
Bundle nmrbundle_data/Bundle_A.pdb Coupling: C3-H3P
Bundle nmrbundle_data/Bundle_A.pdb Coupling: C3-2H5P

# Loading nmrbundle_data/Bundle_Farfar1.pdb 
# Loading nmrbundle_data/Bundle_Farfar2.pdb 


Calculated jcouplings for ['H3H4', 'H3P', '2H5P', 'H1H2', 'C4Pe', '1H5P', 'H2H3'].
(20, 14, 7)
(14, 7)
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G1-H3H4
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G1-H3P
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G1-2H5P
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G1-H1H2
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G1-C4Pe
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G1-1H5P
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G1-H2H3
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G2-H3H4
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G2-H3P
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G2-2H5P
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G2-H1H2
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G2-C4Pe
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G2-1H5P
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: G2-H2H3
Bundle nmrbundle_data/Bundle_Farfar1.pdb Coupling: C3-H3H4
Bundle nmrbund

# Loading nmrbundle_data/Bundle_alphafold3.pdb 


3J's of 2F87 needs to be calculated differently due to different closing basepair

In [18]:
j3_exp_2F87_labels = list(pd.read_csv('exp_data/exp_j3_2F87_bme.dat', delim_whitespace=True).index) # ONLY EXTENDED-LOOP RESIDUES!!!
j3_exp_2F87_couplings = list(set([ i.split('-')[-1] for i in j3_exp_2F87_labels ]))

for bundle in bundles[-1:]:  #2F87 is calculated here
    traj_file   = f'{bundles_dir}/Bundle_{bundle}.pdb'
    top_file    = f'{bundles_dir}/Bundle_{bundle}.pdb'

    couplings,residues = bb.jcouplings(traj_file, topology=top_file, couplings=j3_exp_2F87_couplings)
    print( f'Calculated jcouplings for {j3_exp_2F87_couplings}.' )
    print( couplings.shape )

    couplings_avg = np.average( couplings, 0 )
    print(couplings_avg.shape)
    
    j3_calc_ = pd.DataFrame()
    for i,ii in enumerate([ i.split('_')[0]+i.split('_')[1] for i in residues]):
        for j,jj in enumerate(j3_exp_2F87_couplings):
            j3_calc_[f'{ii}-{jj}'] = couplings[:,i,j]
    
    ll = []
    for i in j3_exp_2F87_labels:
        ll.append(j3_calc_[i])
    pd.DataFrame(ll).T.to_csv(f'{bundles_dir}/nmrbundle_{bundle}_j3_loop_bme.dat', sep=' ', header=False)

Calculated jcouplings for ['H3H4', 'H3P', '2H5P', 'H1H2', 'C4Pe', '1H5P', 'H2H3'].
(13, 12, 7)
(12, 7)


c:\Users\david_leopold\anaconda3\Lib\site-packages\mdtraj\formats\pdb\pdbfile.py:200: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn('Unlikely unit cell vectors detected in PDB file likely '
# Loading nmrbundle_data/Bundle_2F87.pdb 


## CCRs

We use the script ```calc_ccr.py``` to back-calculate CCRs from pdb files or trajectories
Important notes:
- The PDB naming has to be in a specific format (check the script if in doubt).
- Here, $\Gamma$-HCP (C4p-P, C3p-P-plus, C4p-P-plus) and $\Gamma$-HCNCH (C1-CC) are measured at 600 MHz (14.09 T) while $\Gamma$-HCCH (C1-C2, C3-C4) is measured at 700 MHz (16.44 T) which means that in theory the script has to be executed twice and B0 has to be adjusted accordingly. However, $\Gamma$-HCCH couplings are not influenced by B0 which is why we can ignore this for now.

Calculate CCRs from bundles:

Note: certain atoms need specific naming in the .pdb files. Change manualy if necessary 

In [ ]:
# make sure the pdbs use OP1 and OP2 instead of O1P and O2P
ccrs = {"HCCH":14.09,
        "HCN":16.44,
        "HCP":22.30}

for ccr in ccrs:
    field = ccrs[ccr]
    # Check the script and change output name etc.
    for bundle in bundles:
        process = Popen(f'python {bundles_dir}/calc_ccr_2.py {bundles_dir}/Bundle_{bundle}_ccr.pdb "{field}"', shell=True, stdin=PIPE, stdout=PIPE, universal_newlines=True)
        process.wait() # needed to run one process at a time (because of using temporary files etc.

In [15]:
for ccr in ccrs:
    field = ccrs[ccr]
#    ccr_exp_loop  = np.loadtxt(f'exp_data/exp_ccr_{ccr}.dat', usecols=[1,2])
#    ccr_exp_loop_2F87  = np.loadtxt(f'exp_data/exp_ccr_{ccr}_2F87_loop_bme.dat', usecols=[1,2])
    ccr_exp_labels_loop  = list(pd.read_csv(f'exp_data/exp_ccr_{ccr}.dat', delim_whitespace=True, usecols=[0]).index)
    ccr_exp_labels_loop_2F87  = list(pd.read_csv(f'exp_data/exp_ccr_{ccr}_2F87_loop_bme.dat', delim_whitespace=True, usecols=[0]).index)
    for bundle in bundles:
        if bundle == "2F87":
                tmp_ccr = pd.read_csv(f'{bundles_dir}/Bundle_{bundle}_ccr_{field}_calc.dat', delim_whitespace=True, index_col=0)
                tmp_ccr_loop = tmp_ccr[ccr_exp_labels_loop_2F87]
        else:
                tmp_ccr = pd.read_csv(f'{bundles_dir}/Bundle_{bundle}_ccr_{field}_calc.dat', delim_whitespace=True, index_col=0)
                tmp_ccr_loop = tmp_ccr[ccr_exp_labels_loop]
                # Check if all data points are contained in the bundle:
                print(list(tmp_ccr_loop.columns))
                # Save as BME readable file:
        tmp_ccr_loop.to_csv(f'{bundles_dir}/nmrbundle_{bundle}_ccr_calc_{ccr}_loop_bme.dat', header=False, sep=' ')

['C5:C1-C2', 'G6:C1-C2', 'A7:C1-C2', 'A8:C1-C2', 'G9:C1-C2', 'G10:C1-C2', 'C5:C3-C4', 'G6:C3-C4', 'A7:C3-C4', 'A8:C3-C4', 'G9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'A7:C1-C2', 'A8:C1-C2', 'G9:C1-C2', 'G10:C1-C2', 'C5:C3-C4', 'G6:C3-C4', 'A7:C3-C4', 'A8:C3-C4', 'G9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'A7:C1-C2', 'A8:C1-C2', 'G9:C1-C2', 'G10:C1-C2', 'C5:C3-C4', 'G6:C3-C4', 'A7:C3-C4', 'A8:C3-C4', 'G9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'A7:C1-C2', 'A8:C1-C2', 'G9:C1-C2', 'G10:C1-C2', 'C5:C3-C4', 'G6:C3-C4', 'A7:C3-C4', 'A8:C3-C4', 'G9:C3-C4', 'G10:C3-C4']
['C5:C1-CC', 'A7:C1-CC', 'A8:C1-CC', 'G9:C1-CC', 'G10:C1-CC']
['C5:C1-CC', 'A7:C1-CC', 'A8:C1-CC', 'G9:C1-CC', 'G10:C1-CC']
['C5:C1-CC', 'A7:C1-CC', 'A8:C1-CC', 'G9:C1-CC', 'G10:C1-CC']
['C5:C1-CC', 'A7:C1-CC', 'A8:C1-CC', 'G9:C1-CC', 'G10:C1-CC']
['C5:C4p-P', 'G6:C4p-P', 'G9:C4p-P', 'G10:C4p-P', 'C5:C3p-P-plus', 'C5:C4p-P-plus', 'G9:C4p-P-plus', 'G10:C4p-P-plus']
['C5:C4p-P', 'G6:C4p-P', 'G9:C4p-P', 'G10:C4p

## NOEs

We load NOEs from an ARIA output file after they have been converted to distances (\AA). For this, we use the NOEs from bundle F because those have been restraint the most by other types of data. Generally, they don't differ significantly between the different bundles.

Load and parse experimental NOE distance file:

In [6]:
seq = {1:'G', 2:'G', 3:'C', 4:'A', 5:'C', 6:'G', 7:'A', 8:'A', 9:'G', 10:'G', 11:'U', 12:'G', 13:'C', 14:'C'}
for bundle in bundles:    
    df_noe_exp = pd.DataFrame()
    
    with open(f'exp_data/exp_noe.tbl') as f: 
        
        for heading_and_lines in group_by_heading( f ):
            heading= heading_and_lines[0]
            lines= heading_and_lines[1:]
            
            resid1 = int(re.search('resid (\d+)', lines[0] ).group(1))
            name1 = re.search('name (.+)', lines[0] ).group(1).strip()
            resid2 = int(re.search('resid (\d+)', lines[1] ).group(1))
            name2 = re.search('name (.+)', lines[1] ).group(1).strip()
            
            row = {'Assignment': f'{seq[resid1]}{resid1}{name1}-{seq[resid2]}{resid2}{name2}', 'distance':float(lines[2].split()[0]), 'upper_err':float(lines[2].split()[1]), 'lower_err':float(lines[2].split()[2])}
            df_row = pd.DataFrame.from_dict(row, orient='index')
            df_noe_exp = pd.concat([df_noe_exp, df_row], axis=1)
        df_noe_exp = df_noe_exp.T

Make separate dfs for loop NOEs:

In [7]:
residues = ['G1', 'G2', 'C3', 'A4', 'C5', 'G6', 'A7', 'A8', 'G9', 'G10', 'U11', 'G12', 'C13', 'C14']
loop_c5 = ['C5','G6', 'A7', 'A8', 'G9','G10']
loop_less = ['G6', 'A7', 'A8', 'G9','G10']


def create_df_noe_exp_loop(loop):
    a_loop = []
    for i,n in df_noe_exp[:].iterrows():
        a = n['Assignment'].split('-')
        r1 = re.split('(^[A-Z]\d+)', a[0])[1]
        r2 = re.split('(^[A-Z]\d+)', a[1])[1]

        if r1 in loop and r2 in loop:
            a_loop.append(n['Assignment'])
            print(f"Added Assignment: {n['Assignment']}")

        # elif r1 in loop or r2 in loop:
        #     a_both.append(n['Assignment'])
        # else:
        #     a_stem.append(n['Assignment'])
    print(f'Loop NOEs: {len(a_loop)}')
    # print( f'Loop NOEs: {len(a_loop)}\nStem NOEs: {len(a_stem)}\nIntersect: {len(a_both)}' )
    return df_noe_exp.loc[df_noe_exp.Assignment.isin(a_loop)], a_loop

df_noe_exp_loop, a_loop = create_df_noe_exp_loop(loop_c5)
df_noe_exp_loop_less, a_loop_less = create_df_noe_exp_loop(loop_less)
df_noe_exp_all, all = create_df_noe_exp_loop(residues)

print("shape",df_noe_exp_loop.shape)
print("shape",df_noe_exp_loop_less.shape)
print("shape",df_noe_exp_all.shape)

Added Assignment: A7H5'1-A8H8
Added Assignment: A7H2'-A8H8
Added Assignment: C5H5-C5H1'
Added Assignment: C5H5'2-C5H1'
Added Assignment: C5H5'2-C5H5
Added Assignment: G6H3'-G6H1'
Added Assignment: G6H5'1-G6H1'
Added Assignment: G6H5'2-G6H1'
Added Assignment: A7H2'-A7H1'
Added Assignment: A7H5'2-A7H1'
Added Assignment: A7H5'2-A7H5'1
Added Assignment: A8H5'2-A8H1'
Added Assignment: G9H2'-G9H3'
Added Assignment: G9H3'-A8H2
Added Assignment: G9H4'-G9H3'
Added Assignment: G9H5'1-G9H3'
Added Assignment: G10H4'-G9H1'
Added Assignment: A7H4'-A7H1'
Added Assignment: A7H2'-A7H8
Added Assignment: C5H1'-G6H1'
Added Assignment: C5H2'-C5H5
Added Assignment: C5H3'-C5H6
Added Assignment: G6H1'-A7H8
Added Assignment: G6H3'-A7H8
Added Assignment: G6H5'1-A7H8
Added Assignment: A7H5'1-A8H8
Added Assignment: A7H5'2-G6H1'
Added Assignment: A8H4'-A7H1'
Added Assignment: A8H4'-A7H2
Added Assignment: A8H5'1-A8H1'
Added Assignment: A8H5'2-A8H5'1
Added Assignment: A8H5'2-G9H8
Added Assignment: G9H3'-A8H1'
Added 

Write BME readable file for the experimental loop noes:

In [8]:
df_noe_exp_loop_tobme = df_noe_exp_loop.copy().drop(columns=['lower_err', 'upper_err'])
df_noe_exp_loop_tobme['avg_err'] = ((df_noe_exp_loop['lower_err']+df_noe_exp_loop['upper_err'])/2)

df_noe_exp_all_tobme = df_noe_exp_all.copy().drop(columns=['lower_err', 'upper_err'])
df_noe_exp_all_tobme['lower_err'] = (df_noe_exp_all['lower_err'])
df_noe_exp_all_tobme['upper_err'] = (df_noe_exp_all['upper_err'])

In [9]:
with open(f'exp_data/exp_noe_loop_bme.dat', 'w') as f:
    f.write('# DATA=NOE POWER=6\n')
    df_noe_exp_loop_tobme.to_csv(f, index= False, header = False, sep = '\t', lineterminator='\n', float_format='%.2f') #df_noe_exp_tobme[df_noe_exp_tobme['Assignment'].isin(a_loop)]

with open(f'exp_data/exp_noe_all_SI.dat', 'w') as f:
    f.write('# DATA=NOE POWER=6\n')
    df_noe_exp_all_tobme.to_csv(f, index= False, header = False, sep = '\t', lineterminator='\n', float_format='%.2f') 

In [40]:
# Calc distances from the bundles (2F87 has to be one separately due to missing/different residues)
for bundle in bundles[:-1]: #all except last 2F87
    print(bundle)
    traj_ = md.load_pdb(f'nmrbundle_data/Bundle_{bundle}.pdb')
    # labels = get_labels(df_noe_exp, 0) # For all NOEs
    labels = get_labels(df_noe_exp_loop, 0)
    pairs = get_idxs( labels, traj_.topology )
    #print(f'labels:\n{labels}')
    #print(f'pairs:\n{pairs}')
    # calculate distances multiply by 10 to convert to angs
    dists = 10.0*md.compute_distances(traj_,pairs)
    df_noe = pd.DataFrame(dists)
    
    df_noe.columns = list(df_noe_exp_loop.Assignment)
    # Save to BME format:
    print(df_noe.shape)
    df_noe.to_csv(f'nmrbundle_data/nmrbundle_{bundle}_dists_loop.dat', sep='\t', header=False)

A
(20, 135)
Farfar1
(20, 135)
Farfar2
(10, 135)
alphafold3
(5, 135)


In [42]:
# Calc distances from the bundles (2F87 has to be one separately due to missing/different residues)
for bundle in bundles[-1:]: #all except last 2F87
    print(bundle)
    traj_ = md.load_pdb(f'nmrbundle_data/Bundle_{bundle}.pdb')
    # labels = get_labels(df_noe_exp, 0) # For all NOEs
    labels = get_labels(df_noe_exp_loop_less, 0)
    pairs = get_idxs( labels, traj_.topology )
    #print(f'labels:\n{labels}')
    #print(f'pairs:\n{pairs}')
    # calculate distances multiply by 10 to convert to angs
    dists = 10.0*md.compute_distances(traj_,pairs)
    df_noe = pd.DataFrame(dists)
    
    df_noe.columns = list(df_noe_exp_loop_less.Assignment)
    # Save to BME format:
    print(df_noe.shape)
    df_noe.to_csv(f'nmrbundle_data/nmrbundle_{bundle}_dists_loop.dat', sep='\t', header=False)

2F87
(13, 108)


c:\Users\david_leopold\anaconda3\Lib\site-packages\mdtraj\formats\pdb\pdbfile.py:200: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn('Unlikely unit cell vectors detected in PDB file likely '


# From MD ensembles

## RDCs

We use the pf1-phage prediction method implemented in *PALES** to back-calculate RDCs. In order to do that we use the script ```calc_rdc_traj.py``` which submits each frame to *PALES*, predicts the alignment tensor and calculates RDCs, parses the output and saves the D values in a Pickle file.
Just run the code block! Script is now imported.

The actual command line to run *PALES is*: ```pales-linux -inD exp_data/exp_rdc_pales.tab -pdb {pdb_tmp} -outD {outd_tmp} -pf1 -H -wv 0.05```


*Zweckstetter, M. NMR: Prediction of molecular alignment from structure using the PALES software. Nat. Protoc. 3, 679–690 (2008).

In [ ]:
# After running the script above (calc_rdc_traj_.py):
df = load(f'gaag_simulations/d_df_pf1.pkl') # Change


# All to BME
df.T.to_csv(f'calc_data/calc_rdc_bme.dat', header=False, sep='\t')
# Loop to BME:
df.T[rdc_exp_labels_loop].to_csv(f'calc_data/calc_rdc_loop_bme.dat', header=False, sep='\t')

## $^3$J-couplings

We use Barnaba* to calculate the $^3$J scalar couplings 2H5H4, H1H2, H2H3, H3P, C4Pb, 1H5P, 1H5H4, C4Pe, 2H5P and H3H4. (From simulations)

Definition: $A cos^2 (\theta + \phi) + B cos (\theta + \phi) + C$

*Bottaro, S. et al. Barnaba: software for analysis of nucleic acid structures and trajectories. Rna 25, 219–231 (2019)

In [ ]:
# 3J couplings
j3_exp_labels = list(pd.read_csv('exp_data/exp_j3_bme.dat', delim_whitespace=True).index) # ONLY EXTENDED-LOOP RESIDUES!!!
j3_exp_couplings = list(set([ i.split('-')[-1] for i in j3_exp_labels ]))
###

#Never used?
barnaba_to_exp = {'H1H2':"H1',H2'", 'H2H3':"H2',H3'", 'H3H4':"H3',H4'",                                                    
                '1H5P':"H5'i,Pi", '2H5P':"H5''i,Pi",                                                        
                'C4Pb':"C4'i,Pi", '1H5H4':"NA", '2H5H4':"NA",                                                            
                'H3P':"H3'i,Pi+1" ,'C4Pe':"C4'i,Pi+1",                                                                   
                'H1C2/4':"NA", 'H1C6/8':"NA"}                                                                               
###
###
# the shape of couplings is (nframes, nresidues, ncouplings)
###
traj_file   = f'gaag_simulations/concat_traj_nopbc.xtc'
top_file    = f'gaag_simulations/initial_nopbc_mdtraj_ccr.pdb'
###
couplings,residues = bb.jcouplings(traj_file, topology=top_file, couplings=j3_exp_couplings)
print( f'Calculated jcouplings for {j3_exp_couplings}.' )
print( couplings.shape )
###
couplings_avg = np.average( couplings, 0 )
print(couplings_avg.shape)
###
j3_calc_ = pd.DataFrame()
for i,ii in enumerate([ i.split('_')[0]+i.split('_')[1] for i in residues]):
    for j,jj in enumerate(j3_exp_couplings):
        j3_calc_[f'{ii}-{jj}'] = couplings[:,i,j]
###
ll = []
for i in j3_exp_labels:
    ll.append(j3_calc_[i])
pd.DataFrame(ll).T.to_csv(f'calc_data/calc_j3_bme.dat', sep=' ', header=False)

# Loading gaag_simulations/concat_traj_nopbc.xtc 


Calculated jcouplings for ['H3H4', 'H3P', '2H5P', 'H1H2', 'C4Pe', '1H5P', 'H2H3'].
(20100, 14, 7)
(14, 7)
label C5-H1H2 0        0.576833
1        0.013416
2        1.169740
3        0.927197
4       -0.041878
           ...   
20095    2.443970
20096    0.600147
20097    0.504906
20098    0.852262
20099    0.577021
Name: C5-H1H2, Length: 20100, dtype: float64
label G6-H1H2 0        0.628010
1        0.894677
2        0.391928
3        1.438967
4        0.088486
           ...   
20095    0.562312
20096    0.166418
20097    0.575184
20098    0.143887
20099   -0.096189
Name: G6-H1H2, Length: 20100, dtype: float64
label A7-H1H2 0        2.188040
1       -0.089914
2        0.738860
3       -0.035243
4        0.342141
           ...   
20095    1.082161
20096    0.105980
20097   -0.023916
20098    0.003873
20099    0.995159
Name: A7-H1H2, Length: 20100, dtype: float64
label A8-H1H2 0        0.033114
1        0.649862
2       -0.101342
3        0.521326
4        0.844493
           ...   
2

## CCRs

We use the script ```calc_ccr.py``` to back-calculate CCRs from pdb files or trajectories
Important notes:
- The PDB naming has to be in a specific format (check the script if in doubt).
- Here, $\Gamma$-HCP (C4p-P, C3p-P-plus, C4p-P-plus) and $\Gamma$-HCNCH (C1-CC) are measured at 600 MHz (14.09 T) while $\Gamma$-HCCH (C1-C2, C3-C4) is measured at 700 MHz (16.44 T) which means that in theory the script has to be executed twice and B0 has to be adjusted accordingly. However, $\Gamma$-HCCH couplings are not influenced by B0 which is why we can ignore this for now.

In [ ]:
# make sure to select the correct B0 in the calc_ccr.py file
ccrs = ["HCN"]
for ccr in ccrs:
    ccr_exp_labels_loop  = list(pd.read_csv(f'exp_data/exp_ccr_{ccr}.dat', delim_whitespace=True, usecols=[0]).index)

    # Check the script and change output name etc.
    folder_ccr = "calc_data"
    traj_file   = f'gaag_simulations/concat_traj_nopbc.xtc'             # due to the nature of calc_ccr.py and its application for nmr_bundles, the output file will be not in the calc_Data folder and needs renaming 
    top_file    = f'gaag_simulations/initial_nopbc_mdtraj_ccr.pdb'      # due to the nature of calc_ccr.py and its application for nmr_bundles, the output file will be not in the calc_Data folder and needs renaming 

    from importlib import reload
    import nmrbundle_data.calc_ccr as ccr_calc
    # Reload if changes were made on the fly
    reload(ccr_calc)

    traj = md.load(traj_file, top=top_file)

    atom_renaming = {
                    "H5'2":"2H5'",
                    "H5'1":"1H5'",
                    # "H2'1":"H2'",
                    "H2'":"1H2'",

                        }


    tmp_ccr = ccr_calc.calc(traj, atom_renaming)
    tmp_ccr_loop = tmp_ccr[ccr_exp_labels_loop]
    # Check if all data points are contained in the bundle:
    print(list(tmp_ccr_loop.columns))
    # Combine 700 MHz data for gamma-HCCH (C1-C2, C3-C4) couplings with the rest at 600 MHz:
    # Save as BME readable file:
    tmp_ccr_loop.to_csv(f'{folder_ccr}/calc_ccr_{ccr}_loop_bme.dat', header=False, sep=' ', float_format='%8.4e')

renaming G1-H5'1 to G1-1H5'
renaming G1-H5'2 to G1-2H5'
renaming G1-H2' to G1-1H2'
renaming G2-H5'1 to G2-1H5'
renaming G2-H5'2 to G2-2H5'
renaming G2-H2' to G2-1H2'
renaming C3-H5'1 to C3-1H5'
renaming C3-H5'2 to C3-2H5'
renaming C3-H2' to C3-1H2'
renaming A4-H5'1 to A4-1H5'
renaming A4-H5'2 to A4-2H5'
renaming A4-H2' to A4-1H2'
renaming C5-H5'1 to C5-1H5'
renaming C5-H5'2 to C5-2H5'
renaming C5-H2' to C5-1H2'
renaming G6-H5'1 to G6-1H5'
renaming G6-H5'2 to G6-2H5'
renaming G6-H2' to G6-1H2'
renaming A7-H5'1 to A7-1H5'
renaming A7-H5'2 to A7-2H5'
renaming A7-H2' to A7-1H2'
renaming A8-H5'1 to A8-1H5'
renaming A8-H5'2 to A8-2H5'
renaming A8-H2' to A8-1H2'
renaming G9-H5'1 to G9-1H5'
renaming G9-H5'2 to G9-2H5'
renaming G9-H2' to G9-1H2'
renaming G10-H5'1 to G10-1H5'
renaming G10-H5'2 to G10-2H5'
renaming G10-H2' to G10-1H2'
renaming U11-H5'1 to U11-1H5'
renaming U11-H5'2 to U11-2H5'
renaming U11-H2' to U11-1H2'
renaming G12-H5'1 to G12-1H5'
renaming G12-H5'2 to G12-2H5'
renaming G12-H2

## NOEs

In [10]:
dfs_noe = {}
#print(df_noe_exp_loop)
labels = get_labels(df_noe_exp_loop, 0)
print(labels)

traj_file   = f'gaag_simulations/concat_traj_nopbc.xtc'
top_file    = f'gaag_simulations/initial_nopbc_mdtraj_ccr.pdb'

traj  = md.load( traj_file, top = top_file ) # load traj
pairs = get_idxs( labels, traj.topology )

print(traj)
print(pairs)
# calculate distances multiply by 10 to convert to angs
dists = 10.0*md.compute_distances(traj,pairs)
df_noe = pd.DataFrame(dists)

df_noe.columns = list(df_noe_exp_loop['Assignment'])
print(df_noe.shape)

df_noe.to_csv(f'calc_data/calc_noe_loop_bme.dat', sep='\t', header=False, float_format='%.2f')

["A7H5'1/A8H8", "A7H2'/A8H8", "C5H1'/C5H5", "C5H1'/C5H5'2", "C5H5/C5H5'2", "G6H1'/G6H3'", "G6H1'/G6H5'1", "G6H1'/G6H5'2", "A7H1'/A7H2'", "A7H1'/A7H5'2", "A7H5'1/A7H5'2", "A8H1'/A8H5'2", "G9H2'/G9H3'", "A8H2/G9H3'", "G9H3'/G9H4'", "G9H3'/G9H5'1", "G10H4'/G9H1'", "A7H1'/A7H4'", "A7H2'/A7H8", "C5H1'/G6H1'", "C5H2'/C5H5", "C5H3'/C5H6", "A7H8/G6H1'", "A7H8/G6H3'", "A7H8/G6H5'1", "A7H5'1/A8H8", "A7H5'2/G6H1'", "A7H1'/A8H4'", "A7H2/A8H4'", "A8H1'/A8H5'1", "A8H5'1/A8H5'2", "A8H5'2/G9H8", "A8H1'/G9H3'", "G10H8/G9H3'", "G9H1'/G9H5'2", "G9H3'/G9H5'2", "G9H5'1/G9H5'2", "G10H2'/G10H8", "C5H3'/C5H5", "C5H3'/G6H8", "G6H4'/G6H8", "A7H2/A7H2'", "A7H5'1/G6H1'", "A7H1'/A8H5'1", "A8H5'1/A8H8", "A7H1'/A8H5'2", "G10H8/G9H1'", "A8H1'/G9H5'2", "G9H5'2/G9H8", "G10H2'/G9H8", "G10H5'1/G10H8", "C5H1'/C5H6", "C5H1'/C5H2'", "C5H2'/C5H6", "C5H2'/G6H1'", "C5H1'/C5H4'", 'C5H41/C5H5', 'C5H5/C5H6', "C5H1'/C5H5'1", "G6H1'/G6H2'", 'A8H8/G6H22', 'G6H22/G9H8', "G6H1'/G6H4'", "G6H5'1/G6H5'2", "A7H1'/A7H2", "A7H1'/A8H8", "A7H